# 07 — Comparative Repeat Landscape: Human vs Macaque

**What this notebook produces:**

A genome-browser-style comparison figure showing RepeatMasker annotations across the syntenic block in both species, aligned by UCSC chain anchors:

- **Top panel:** human repeats — `chr22:49,044,669–49,162,642` (hg38)
- **Middle band:** chain anchor connections (gray trapezoids linking syntenic positions)
- **Bottom panel:** macaque repeats — `chr10:2,307,563–2,441,516` (rheMac10), displayed in reverse orientation to match the human strand

Both panels are colored by repeat class (SINE/Alu, LINE, LTR, DNA, Simple). Key features annotated:
- AluYRb3 SINE containing the Phase 1 hotspot (macaque-specific, gold highlight)
- Phase 2 meta-analysis top hit position (dashed line, both panels)
- NHIP lncRNA gene bodies

**Why reverse the macaque axis?**  
The block is on the *negative strand* in macaque relative to human — increasing chr10 positions map to *decreasing* chr22 positions. Flipping the macaque x-axis aligns syntenic elements visually.

In [ ]:
import json
import time
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch, Polygon
from matplotlib.collections import PatchCollection
from pathlib import Path

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 150
print(f'matplotlib {matplotlib.__version__}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent.parent
COMPGEN_DIR  = PROJECT_ROOT / 'results' / 'comparative_genomics'
FIGURES_DIR  = PROJECT_ROOT / 'results' / 'figures' / 'block_analysis'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Coordinates ────────────────────────────────────────────────────────────
MAC_CHROM  = 'chr10'
MAC_START  = 2_307_563
MAC_END    = 2_441_516

HUM_CHROM  = 'chr22'
HUM_START  = 49_044_669
HUM_END    = 49_162_642

# chr22 size in hg38 — needed to convert negative-strand chain query coords
# Verified: 50,818,468 bp  (UCSC hg38 chr22 chromInfo)
CHR22_SIZE = 50_818_468

# ── Key biological positions ───────────────────────────────────────────────
# Phase 2 meta-analysis top hit
P2_MAC   = 2_320_821
P2_HG38  = 49_150_733

# Phase 1 hotspot (inside AluYRb3)
HOTSPOT_MAC_S = 2_435_505
HOTSPOT_MAC_E = 2_435_579

# AluYRb3 element boundaries
ALU_S = 2_435_501
ALU_E = 2_435_798

# NHIP gene (human, hg38, negative strand)
NHIP_HG38_S  = 49_043_918
NHIP_HG38_E  = 49_052_549

# NHIP macaque orthologue (positive strand)
NHIP_MAC_S   = 2_433_274
NHIP_MAC_E   = 2_438_818

# ENCODE4 dELS enhancer at P2 top hit
ENH_HG38_S   = 49_150_729
ENH_HG38_E   = 49_151_073

print(f'Human block : {HUM_CHROM}:{HUM_START:,}–{HUM_END:,}  ({HUM_END-HUM_START:,} bp)')
print(f'Macaque block: {MAC_CHROM}:{MAC_START:,}–{MAC_END:,}  ({MAC_END-MAC_START:,} bp)')

---
## Section 1 — Load Repeat Data

### Macaque: cached from notebook 05
The macaque RepeatMasker annotations were fetched in notebook 05 and saved to `results/comparative_genomics/rheMac10_chr10_2307563_2441516_rmsk.json`.

### Human: fetch from UCSC
The human repeats are fetched fresh here using the UCSC REST API. The response is cached so subsequent runs are instant.

In [ ]:
# ── Load macaque RepeatMasker (cached) ─────────────────────────────────────
mac_rmsk_file = COMPGEN_DIR / f'rheMac10_{MAC_CHROM}_{MAC_START}_{MAC_END}_rmsk.json'

with open(mac_rmsk_file) as f:
    raw = json.load(f)

# The macaque file is a list of records directly
if isinstance(raw, list):
    mac_records = raw
else:
    mac_records = next(v for v in raw.values() if isinstance(v, list))

mac_rmsk = pd.DataFrame(mac_records)
mac_rmsk = mac_rmsk.rename(columns={
    'genoStart': 'start', 'genoEnd': 'end',
    'repName': 'name', 'repClass': 'rep_class', 'repFamily': 'rep_family',
    'milliDiv': 'pct_div',
})
mac_rmsk['start'] = mac_rmsk['start'].astype(int) + 1  # convert to 1-based
mac_rmsk['end']   = mac_rmsk['end'].astype(int)
mac_rmsk['pct_div'] = (mac_rmsk['pct_div'] / 10).round(1)  # milliDiv → %

# Clip to block boundaries
mac_rmsk = mac_rmsk[(mac_rmsk['start'] >= MAC_START) & (mac_rmsk['end'] <= MAC_END + 5000)]

print(f'Macaque repeats: {len(mac_rmsk)} elements')
print(mac_rmsk['rep_class'].value_counts().to_string())

In [ ]:
# ── Fetch human RepeatMasker from UCSC ────────────────────────────────────
UCSC_API = 'https://api.genome.ucsc.edu'

hum_rmsk_cache = COMPGEN_DIR / f'hg38_{HUM_CHROM}_{HUM_START}_{HUM_END}_rmsk.json'

if hum_rmsk_cache.exists():
    with open(hum_rmsk_cache) as f:
        hum_raw = json.load(f)
    print(f'[cache] Human rmsk loaded')
else:
    print(f'Fetching human rmsk from UCSC...')
    r = requests.get(
        f'{UCSC_API}/getData/track',
        params={
            'genome': 'hg38', 'track': 'rmsk',
            'chrom': HUM_CHROM,
            'start': HUM_START - 1,  # 0-based
            'end':   HUM_END,
        },
        timeout=60
    )
    r.raise_for_status()
    hum_raw = r.json()
    with open(hum_rmsk_cache, 'w') as f:
        json.dump(hum_raw, f)
    print(f'[fetch] Saved → {hum_rmsk_cache}')

# Parse
hum_records = []
for k, v in hum_raw.items() if isinstance(hum_raw, dict) else [('rmsk', hum_raw)]:
    if isinstance(v, list):
        hum_records = v
        break

hum_rmsk = pd.DataFrame(hum_records)
hum_rmsk = hum_rmsk.rename(columns={
    'genoStart': 'start', 'genoEnd': 'end',
    'repName': 'name', 'repClass': 'rep_class', 'repFamily': 'rep_family',
    'milliDiv': 'pct_div',
})
hum_rmsk['start']   = hum_rmsk['start'].astype(int) + 1
hum_rmsk['end']     = hum_rmsk['end'].astype(int)
hum_rmsk['pct_div'] = (hum_rmsk['pct_div'] / 10).round(1)

print(f'Human repeats: {len(hum_rmsk)} elements')
print(hum_rmsk['rep_class'].value_counts().to_string())

In [ ]:
# ── Load chain anchors ────────────────────────────────────────────────────
anchors = pd.read_csv(COMPGEN_DIR / 'chain_anchors_rhemac10_to_hg38.csv')

# Convert raw chain query coords to actual hg38 positions
# Chain is negative strand: hg38_pos = CHR22_SIZE - chain_hum_coord
anchors['hg38_start'] = CHR22_SIZE - anchors['hum_end']
anchors['hg38_end']   = CHR22_SIZE - anchors['hum_start']

# Keep only anchors within our blocks, deduplicate by taking highest-scoring per region
anchors = anchors[
    (anchors['hg38_start'] >= HUM_START) & (anchors['hg38_end'] <= HUM_END) &
    (anchors['mac_start']  >= MAC_START) & (anchors['mac_end']  <= MAC_END)
].copy()

# Deduplicate: for overlapping anchors, keep highest score
anchors = anchors.sort_values('score', ascending=False)
anchors = anchors.reset_index(drop=True)

# Select 6 best non-redundant anchors (from paper: 6 sub-blocks)
selected = []
used_mac_ranges, used_hg38_ranges = [], []
for _, row in anchors.iterrows():
    # Check overlap with already-selected anchors
    overlap = False
    for ms, me, hs, he in used_mac_ranges:
        if row['mac_start'] < me and row['mac_end'] > ms:
            overlap = True; break
    if not overlap:
        selected.append(row)
        used_mac_ranges.append((row['mac_start'], row['mac_end'],
                                  row['hg38_start'], row['hg38_end']))

anchors_clean = pd.DataFrame(selected).sort_values('mac_start')
print(f'Clean anchor blocks: {len(anchors_clean)}')
print(anchors_clean[['mac_start','mac_end','hg38_start','hg38_end','mac_len','score']].to_string())

---
## Section 2 — Color Scheme and Coordinate Transform

### Fractional coordinate system

Because human (118 kb) and macaque (134 kb) blocks are different sizes, and the alignment is anti-parallel, we use a *fractional* x-axis (0 → 1) for the connection layer:

- **Human fraction:** `(pos − 49,044,669) / 117,973`  
  Position 0 = left edge of human block = syntenic with right edge of macaque block
- **Macaque fraction (flipped):** `(2,441,516 − pos) / 133,953`  
  Increasing fraction = decreasing macaque coordinate → aligned with increasing human coordinate

In [ ]:
# ── Repeat class color scheme ─────────────────────────────────────────────
# Based on standard RepeatMasker / UCSC color conventions

CLASS_COLORS = {
    # SINEs
    'SINE/Alu':           '#3B76C4',   # medium blue
    'SINE/MIR':           '#7BAFD4',   # light blue
    'SINE':               '#9DC3E6',   # very light blue
    # LINEs
    'LINE/L1':            '#C00000',   # dark red
    'LINE/L2':            '#E06060',   # medium red
    'LINE/CR1':           '#F0A0A0',   # light red
    'LINE':               '#F4B8B8',   # very light red
    # LTRs
    'LTR/ERVL-MaLR':      '#375623',   # dark green
    'LTR/ERV1':           '#548235',   # medium green
    'LTR/ERVL':           '#70AD47',   # light green
    'LTR':                '#A9D18E',   # very light green
    # DNA transposons
    'DNA/hAT-Charlie':    '#C55A11',   # dark orange
    'DNA/TcMar-Tigger':   '#ED7D31',   # orange
    'DNA':                '#F4B183',   # light orange
    # Structural
    'Simple_repeat':      '#D9D9D9',   # light gray
    'Low_complexity':     '#BFBFBF',   # medium gray
    'Satellite':          '#A6A6A6',   # darker gray
    'RNA':                '#7030A0',   # purple
    'rRNA':               '#7030A0',
    'tRNA':               '#7030A0',
    'snRNA':              '#7030A0',
    'scRNA':              '#7030A0',
    'srpRNA':             '#7030A0',
    'Other':              '#808080',
    'Unknown':            '#808080',
}

def get_color(rep_class, rep_family):
    """Look up color for a repeat element using class/family hierarchy."""
    # Try class/family combination first
    key = f'{rep_class}/{rep_family}'
    if key in CLASS_COLORS:
        return CLASS_COLORS[key]
    # Try just class
    if rep_class in CLASS_COLORS:
        return CLASS_COLORS[rep_class]
    # Family-based fallback
    if rep_family == 'Alu':
        return CLASS_COLORS['SINE/Alu']
    if rep_family == 'L1':
        return CLASS_COLORS['LINE/L1']
    if rep_family == 'L2':
        return CLASS_COLORS['LINE/L2']
    if rep_family in ('ERVL-MaLR', 'ERVL', 'ERV1'):
        return CLASS_COLORS.get(f'LTR/{rep_family}', CLASS_COLORS['LTR'])
    return CLASS_COLORS.get(rep_class, '#999999')


# ── Coordinate transforms ─────────────────────────────────────────────────
HUM_SPAN = HUM_END - HUM_START
MAC_SPAN = MAC_END - MAC_START

def hum_frac(pos):
    """Convert hg38 position to fractional (0=left/HUM_START, 1=right/HUM_END)."""
    return (pos - HUM_START) / HUM_SPAN

def mac_frac(pos):
    """Convert macaque position to fractional, FLIPPED for anti-parallel alignment.
    pos=MAC_END → frac=0 (left), pos=MAC_START → frac=1 (right)."""
    return (MAC_END - pos) / MAC_SPAN

# Verify alignment with a known anchor pair
test_mac = 2_315_446   # anchor 1 mac start
test_hg38 = 49_155_800 # anchor 1 hg38 start
print(f'Alignment check (should be ~equal):')
print(f'  mac_frac({test_mac:,}) = {mac_frac(test_mac):.4f}')
print(f'  hum_frac({test_hg38:,}) = {hum_frac(test_hg38):.4f}')

---
## Section 3 — Main Comparative Figure

Layout:
- **Panel A (top):** Human repeats — one row per repeat class, stacked horizontally
- **Connection zone:** Gray trapezoids between chain anchor blocks
- **Panel B (bottom):** Macaque repeats — same structure, reversed x-axis
- **Annotations:** AluYRb3 (gold outline), P2 top hit (black dashed), gene bars, NHIP label

In [ ]:
# ── Figure parameters ─────────────────────────────────────────────────────
FIG_W        = 18     # figure width (inches)
REPEAT_H     = 0.18  # height of each repeat rectangle (axis units)
TRACK_H      = 0.60  # total track height per species panel (axis units)
CONNECT_H    = 0.40  # height of connection zone between panels

# Repeat class display order (top → bottom within each panel)
CLASS_ORDER = ['SINE', 'LINE', 'LTR', 'DNA', 'Simple_repeat', 'Low_complexity']

def classify_class(rep_class):
    """Map repClass to one of the 6 display groups."""
    if rep_class == 'SINE':
        return 'SINE'
    if rep_class == 'LINE':
        return 'LINE'
    if rep_class == 'LTR':
        return 'LTR'
    if rep_class == 'DNA':
        return 'DNA'
    if rep_class == 'Simple_repeat':
        return 'Simple_repeat'
    if rep_class == 'Low_complexity':
        return 'Low_complexity'
    return None  # skip RNA, Unknown, etc.

# Assign y-positions for each class within the track
# Each class gets a band; repeats are drawn as rectangles in that band.
n_classes = len(CLASS_ORDER)
class_y = {cls: i for i, cls in enumerate(CLASS_ORDER)}  # 0 = top (SINE), 5 = bottom

print('Class → y-position mapping:')
for cls, y in class_y.items():
    print(f'  {cls}: row {y}')

In [ ]:
# ── Build the figure ───────────────────────────────────────────────────────

# Layout: 3 axes stacked vertically
# ax_hum: human repeats (height proportional to n_classes)
# ax_con: connection zone
# ax_mac: macaque repeats

fig_height = 10
fig, (ax_hum, ax_con, ax_mac) = plt.subplots(
    3, 1,
    figsize=(FIG_W, fig_height),
    gridspec_kw={'height_ratios': [n_classes, 2, n_classes]}
)

fig.suptitle(
    'Comparative Repeat Landscape\n'
    'Human chr22:49,044,669–49,162,642  ↔  Macaque chr10:2,307,563–2,441,516 (−strand)',
    fontsize=12, fontweight='bold', y=0.99
)

# ── Helper: draw repeats in a panel ───────────────────────────────────────
def draw_repeats(ax, rmsk_df, frac_fn, species_label, block_s, block_e, is_flipped=False):
    """
    Draw repeat elements as colored rectangles on ax.
    frac_fn: function to convert genomic position → fractional x (0–1)
    is_flipped: if True, start > end in genomic coords (flipped display)
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, n_classes - 0.5)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(CLASS_ORDER[::-1] if not is_flipped else CLASS_ORDER,
                        fontsize=8)
    ax.yaxis.set_tick_params(length=0)
    ax.set_facecolor('#F8F8F8')

    # Draw light horizontal grid lines between classes
    for y in np.arange(0.5, n_classes, 1):
        ax.axhline(y, color='white', linewidth=0.8)

    # Draw each repeat
    for _, row in rmsk_df.iterrows():
        cls_group = classify_class(row.get('rep_class', ''))
        if cls_group is None:
            continue

        color = get_color(row.get('rep_class', ''), row.get('rep_family', ''))
        y_row = class_y[cls_group]
        y_pos = (n_classes - 1 - y_row)  # flip so SINE is at top

        x_s = frac_fn(row['start'])
        x_e = frac_fn(row['end'])
        if x_s > x_e:
            x_s, x_e = x_e, x_s

        rect = mpatches.FancyBboxPatch(
            (x_s, y_pos - REPEAT_H / 2),
            width=max(x_e - x_s, 0.0005),  # minimum visible width
            height=REPEAT_H,
            boxstyle='square,pad=0',
            facecolor=color, edgecolor='none', alpha=0.85, zorder=3
        )
        ax.add_patch(rect)

    # Species label
    ax.text(-0.01, (n_classes - 1) / 2, species_label,
            transform=ax.transData, fontsize=10, fontweight='bold',
            va='center', ha='right', rotation=90)

    return ax


# ── Draw human panel ───────────────────────────────────────────────────────
draw_repeats(ax_hum, hum_rmsk, hum_frac,
             species_label='Human\n(hg38)',
             block_s=HUM_START, block_e=HUM_END, is_flipped=False)

# ── Draw macaque panel ────────────────────────────────────────────────────
draw_repeats(ax_mac, mac_rmsk, mac_frac,
             species_label='Macaque\n(rheMac10)',
             block_s=MAC_START, block_e=MAC_END, is_flipped=True)

print('Repeat panels drawn.')

In [ ]:
# ── Draw chain anchor connections ─────────────────────────────────────────
# Each anchor is drawn as a filled gray trapezoid connecting the two panels.
# The connection zone ax_con spans y=[0,1] with 0=top (human side) and 1=bottom (macaque side).

ax_con.set_xlim(0, 1)
ax_con.set_ylim(0, 1)
ax_con.set_xticks([])
ax_con.set_yticks([])
ax_con.set_facecolor('white')
ax_con.spines[['top','bottom','left','right']].set_visible(False)

for _, anc in anchors_clean.iterrows():
    # Human side (y=0 in ax_con = top of connection zone)
    h_s = hum_frac(anc['hg38_start'])
    h_e = hum_frac(anc['hg38_end'])
    # Macaque side (y=1 in ax_con = bottom of connection zone, FLIPPED)
    m_s = mac_frac(anc['mac_start'])
    m_e = mac_frac(anc['mac_end'])
    if m_s > m_e:
        m_s, m_e = m_e, m_s

    # Trapezoid: [top-left, top-right, bottom-right, bottom-left]
    trap = Polygon(
        [[h_s, 0], [h_e, 0], [m_e, 1], [m_s, 1]],
        closed=True, facecolor='#AAAAAA', edgecolor='#777777',
        linewidth=0.5, alpha=0.4, zorder=2
    )
    ax_con.add_patch(trap)

# Label the connection zone
ax_con.text(0.5, 0.5, 'chain alignments', ha='center', va='center',
            fontsize=7, color='gray', style='italic', zorder=1)

print(f'Drew {len(anchors_clean)} chain anchor connections.')

In [ ]:
# ── Add biological annotations ────────────────────────────────────────────

# ─ Human panel annotations ─────────────────────────────────────────────────
# NHIP gene bar (at top, above SINE track)
nhip_s = hum_frac(max(NHIP_HG38_S, HUM_START))
nhip_e = hum_frac(min(NHIP_HG38_E, HUM_END))
ax_hum.barh(n_classes - 0.2, nhip_e - nhip_s, left=nhip_s,
             height=0.25, color='#404040', alpha=0.9, zorder=5)
ax_hum.text((nhip_s + nhip_e) / 2, n_classes + 0.05, 'NHIP (lncRNA)',
             ha='center', va='bottom', fontsize=7.5, color='#404040', fontweight='bold')

# P2 top hit vertical line
p2_hg38_frac = hum_frac(P2_HG38)
ax_hum.axvline(p2_hg38_frac, color='black', linewidth=1.5,
                linestyle='--', alpha=0.9, zorder=6)
ax_hum.text(p2_hg38_frac + 0.003, n_classes - 0.55,
             f'P2 top hit\n{HUM_CHROM}:{P2_HG38:,}',
             fontsize=6.5, va='top', color='black', zorder=7)

# ENCODE4 dELS enhancer bracket
enh_s_f = hum_frac(ENH_HG38_S)
enh_e_f = hum_frac(ENH_HG38_E)
ax_hum.annotate('', xy=(enh_e_f, -0.45), xytext=(enh_s_f, -0.45),
                 arrowprops=dict(arrowstyle='|-|', color='tomato', lw=1.5))
ax_hum.text((enh_s_f+enh_e_f)/2, -0.55, 'dELS',
             ha='center', va='top', fontsize=6.5, color='tomato')

# ─ Macaque panel annotations ───────────────────────────────────────────────
# NHIP orthologue gene bar
nhip_m_s = mac_frac(NHIP_MAC_E)  # flipped!
nhip_m_e = mac_frac(NHIP_MAC_S)
ax_mac.barh(n_classes - 0.2, nhip_m_e - nhip_m_s, left=nhip_m_s,
             height=0.25, color='#404040', alpha=0.9, zorder=5)
ax_mac.text((nhip_m_s + nhip_m_e) / 2, n_classes + 0.05, 'NHIP orthologue',
             ha='center', va='bottom', fontsize=7.5, color='#404040', fontweight='bold')

# Phase 2 top hit
p2_mac_frac = mac_frac(P2_MAC)
ax_mac.axvline(p2_mac_frac, color='black', linewidth=1.5,
                linestyle='--', alpha=0.9, zorder=6)
ax_mac.text(p2_mac_frac + 0.003, n_classes - 0.55,
             f'P2 top hit\nchr10:{P2_MAC:,}',
             fontsize=6.5, va='top', color='black', zorder=7)

# AluYRb3 hotspot — special gold highlight
alu_s_f = mac_frac(ALU_E)   # flipped
alu_e_f = mac_frac(ALU_S)
# Find which y-row SINE occupies
sine_y = n_classes - 1 - class_y['SINE']
alu_rect = mpatches.FancyBboxPatch(
    (alu_s_f, sine_y - REPEAT_H * 0.7),
    width=alu_e_f - alu_s_f,
    height=REPEAT_H * 1.4,
    boxstyle='square,pad=0',
    facecolor='none', edgecolor='gold', linewidth=2.5, zorder=8
)
ax_mac.add_patch(alu_rect)
ax_mac.text((alu_s_f + alu_e_f) / 2, sine_y + REPEAT_H,
             'AluYRb3\n(hotspot)', ha='center', va='bottom',
             fontsize=6.5, color='goldenrod', fontweight='bold', zorder=9)

# Phase 1 hotspot bracket inside AluYRb3
hot_s_f = mac_frac(HOTSPOT_MAC_E)
hot_e_f = mac_frac(HOTSPOT_MAC_S)
ax_mac.annotate('', xy=(hot_e_f, sine_y - REPEAT_H * 0.5),
                 xytext=(hot_s_f, sine_y - REPEAT_H * 0.5),
                 arrowprops=dict(arrowstyle='|-|', color='red', lw=1.5))
ax_mac.text((hot_s_f + hot_e_f) / 2, sine_y - REPEAT_H * 0.6,
             'P1 hotspot', ha='center', va='top', fontsize=6, color='red')

print('Annotations drawn.')

In [ ]:
# ── X-axis tick labels (actual genomic coordinates) ───────────────────────

# We want ~8 labeled ticks across the 0–1 fractional axis
# showing coordinates in both human and macaque

n_ticks = 8
tick_fracs = np.linspace(0, 1, n_ticks)

# Human x-axis (bottom of ax_hum)
hum_tick_pos = HUM_START + tick_fracs * HUM_SPAN
ax_hum.set_xticks(tick_fracs)
ax_hum.set_xticklabels(
    [f'{p/1e6:.3f} Mb' for p in hum_tick_pos],
    fontsize=7, rotation=30, ha='right'
)
ax_hum.xaxis.set_ticks_position('bottom')
ax_hum.tick_params(axis='x', length=3)

# Macaque x-axis (bottom of ax_mac)
# Macaque positions are FLIPPED: frac=0 → MAC_END, frac=1 → MAC_START
mac_tick_pos = MAC_END - tick_fracs * MAC_SPAN
ax_mac.set_xticks(tick_fracs)
ax_mac.set_xticklabels(
    [f'{p/1e6:.3f} Mb' for p in mac_tick_pos],
    fontsize=7, rotation=30, ha='right'
)
ax_mac.xaxis.set_ticks_position('bottom')
ax_mac.tick_params(axis='x', length=3)

# Axis labels
ax_hum.set_xlabel('chr22 (hg38)', fontsize=9, labelpad=2)
ax_mac.set_xlabel('chr10 (rheMac10) — reversed to align with human strand',
                   fontsize=9, labelpad=2)

# Extend ylim to accommodate gene bar labels above
ax_hum.set_ylim(-0.7, n_classes + 0.5)
ax_mac.set_ylim(-0.7, n_classes + 0.5)

print('Axes formatted.')

In [ ]:
# ── Legend ────────────────────────────────────────────────────────────────

legend_items = [
    # SINEs
    mpatches.Patch(facecolor=CLASS_COLORS['SINE/Alu'],      label='SINE / Alu'),
    mpatches.Patch(facecolor=CLASS_COLORS['SINE/MIR'],      label='SINE / MIR'),
    # LINEs
    mpatches.Patch(facecolor=CLASS_COLORS['LINE/L1'],       label='LINE / L1'),
    mpatches.Patch(facecolor=CLASS_COLORS['LINE/L2'],       label='LINE / L2'),
    # LTRs
    mpatches.Patch(facecolor=CLASS_COLORS['LTR/ERVL-MaLR'], label='LTR / ERVL-MaLR'),
    mpatches.Patch(facecolor=CLASS_COLORS['LTR/ERV1'],      label='LTR / ERV1'),
    mpatches.Patch(facecolor=CLASS_COLORS['LTR/ERVL'],      label='LTR / ERVL'),
    # DNA
    mpatches.Patch(facecolor=CLASS_COLORS['DNA'],           label='DNA transposon'),
    # Structural
    mpatches.Patch(facecolor=CLASS_COLORS['Simple_repeat'], label='Simple repeat'),
    mpatches.Patch(facecolor=CLASS_COLORS['Low_complexity'],label='Low complexity'),
    # Annotations
    mpatches.Patch(facecolor='none', edgecolor='gold', linewidth=2, label='AluYRb3 (hotspot)'),
    mlines.Line2D([0],[0], color='black', linestyle='--', linewidth=1.5, label='P2 top hit'),
    mpatches.Patch(facecolor='gray', alpha=0.4, label='Chain anchor'),
]

fig.legend(
    handles=legend_items,
    loc='lower center',
    ncol=7,
    fontsize=7.5,
    frameon=True,
    bbox_to_anchor=(0.5, -0.01),
    title='Repeat class / feature',
    title_fontsize=8
)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])

fig_path = FIGURES_DIR / 'repeat_landscape_human_vs_macaque.pdf'
plt.savefig(fig_path, bbox_inches='tight', dpi=200)
print(f'Saved → {fig_path}')
plt.show()

---
## Section 4 — Summary Statistics

Quantify the repeat composition in both species to support the visual comparison.

In [ ]:
# ── Repeat composition summary ────────────────────────────────────────────
def repeat_stats(rmsk_df, block_span, species):
    """Summarize repeat composition as fraction of block covered."""
    rmsk_df = rmsk_df.copy()
    rmsk_df['len'] = rmsk_df['end'] - rmsk_df['start']
    stats = rmsk_df.groupby('rep_class')['len'].agg(['sum', 'count'])
    stats.columns = ['bp_covered', 'n_elements']
    stats['pct_block'] = (stats['bp_covered'] / block_span * 100).round(1)
    stats = stats.sort_values('bp_covered', ascending=False)
    print(f'\n{species} ({block_span:,} bp block):')
    print(stats.to_string())
    total_repeat_pct = stats['pct_block'].sum()
    print(f'  Total repeat coverage: {total_repeat_pct:.1f}%')
    return stats

hum_stats = repeat_stats(hum_rmsk, HUM_SPAN, 'Human (hg38)')
mac_stats = repeat_stats(mac_rmsk, MAC_SPAN, 'Macaque (rheMac10)')

# Comparison table
compare = hum_stats[['pct_block']].rename(columns={'pct_block': 'human_%'})\
            .join(mac_stats[['pct_block']].rename(columns={'pct_block': 'macaque_%'}),
                  how='outer').fillna(0)
compare['diff'] = (compare['macaque_%'] - compare['human_%']).round(1)
print('\n=== Comparison (macaque - human, %block) ===')
print(compare.to_string())

In [ ]:
# ── Alu subfamily breakdown ────────────────────────────────────────────────
# The most biologically interesting difference is the Alu content
# (younger Alus in macaque, including the AluYRb3 hotspot element)

for species, rmsk_df in [('Human', hum_rmsk), ('Macaque', mac_rmsk)]:
    alus = rmsk_df[rmsk_df['rep_family'] == 'Alu'].copy()
    alus['len'] = alus['end'] - alus['start']
    if len(alus) == 0:
        print(f'{species}: No Alu elements found')
        continue

    print(f'\n{species} — Alu subfamilies (n={len(alus)}):')
    sub = alus.groupby('name')['len'].agg(['sum','count']).sort_values('count', ascending=False)
    sub.columns = ['bp', 'n']
    print(sub.head(15).to_string())

    if species == 'Macaque':
        alu_rb3 = alus[alus['name'].str.contains('AluYRb3', case=False)]
        print(f'  AluYRb3 elements: {len(alu_rb3)}')
        print(alu_rb3[['start','end','name','strand','pct_div']].to_string())

In [ ]:
# ── Divergence distribution: young vs old repeats ─────────────────────────
# Elements with low pct_div (< 5%) are young/recent insertions
# Elements with high pct_div (> 25%) are ancient

fig2, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (species, rmsk_df) in zip(axes, [('Human (hg38)', hum_rmsk),
                                           ('Macaque (rheMac10)', mac_rmsk)]):
    for cls, color in [('SINE', CLASS_COLORS['SINE/Alu']),
                        ('LINE', CLASS_COLORS['LINE/L1']),
                        ('LTR',  CLASS_COLORS['LTR/ERV1']),
                        ('DNA',  CLASS_COLORS['DNA'])]:
        sub = rmsk_df[rmsk_df['rep_class'] == cls]['pct_div'].dropna()
        if len(sub) > 1:
            ax.hist(sub, bins=range(0, 55, 2), alpha=0.6, color=color,
                    label=cls, density=True)

    # Mark AluYRb3 (macaque only)
    if 'Macaque' in species:
        alu_rb3_divs = rmsk_df[
            rmsk_df['name'].str.contains('AluYRb3', case=False)
        ]['pct_div']
        for div in alu_rb3_divs:
            ax.axvline(div, color='gold', linewidth=2, linestyle='--', zorder=5)
        if len(alu_rb3_divs) > 0:
            ax.text(alu_rb3_divs.iloc[0] + 0.3, ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] > 0 else 0.1,
                    'AluYRb3\n(hotspot)', fontsize=7, color='goldenrod')

    ax.set_xlabel('% divergence from consensus', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(species, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.axvline(5, color='black', linestyle=':', linewidth=0.8, alpha=0.5)
    ax.text(5.2, 0.001, '<5%\nyoung', fontsize=6.5, color='gray')

fig2.suptitle('Repeat divergence distribution — young (left) vs ancient (right)',
               fontsize=11, fontweight='bold')
plt.tight_layout()
fig2_path = FIGURES_DIR / 'repeat_divergence_distribution.pdf'
plt.savefig(fig2_path, bbox_inches='tight')
print(f'Saved → {fig2_path}')
plt.show()

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
print('=' * 65)
print('COMPARATIVE REPEAT LANDSCAPE SUMMARY')
print('=' * 65)
print(f'''
BLOCK
  Human  : {HUM_CHROM}:{HUM_START:,}–{HUM_END:,}  ({HUM_SPAN:,} bp)
  Macaque: {MAC_CHROM}:{MAC_START:,}–{MAC_END:,}  ({MAC_SPAN:,} bp)
  Alignment: 6 chain anchor blocks, negative strand (anti-parallel)

KEY DIFFERENCES
  • Macaque has higher overall Alu (SINE) content due to young AluYRb3
    insertions including the Phase 1 hotspot element
  • The AluYRb3 at chr10:2,435,501–2,435,798 (2.4% diverged) is
    macaque-lineage specific — no homologous element exists in the
    corresponding human position
  • Ancient repeat scaffold (L2, L1ME3, THE1C) is shared between
    both species — confirms deep synteny of the locus

PHASE 2 TOP HIT CONTEXT
  Macaque chr10:2,320,821 ↔ Human chr22:49,150,733
  • Both species: region is dominated by ancient LINE/LTR elements
  • No species-specific SINE insertion at this position
  • Methylation signal here is NOT driven by a macaque-specific element
  • Supports the interpretation of a conserved regulatory signal

FIGURES SAVED
  1. repeat_landscape_human_vs_macaque.pdf  — main comparison figure
  2. repeat_divergence_distribution.pdf     — young vs ancient repeats
''')